In [5]:
!pip install mlflow xgboost imbalanced-learn

import mlflow
import pandas as pd
import numpy as np
from scipy.sparse import hstack

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

from sklearn.metrics import (
    r2_score, accuracy_score, confusion_matrix,
    mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error,
    precision_score, recall_score, roc_auc_score, f1_score
)

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
df = pd.read_csv("G:/Mohanraj D_OFFICIAL/GUVI (Data Analyst)/Mini-Projects/EMI_Prediction/cleaned_emi_prediction_dataset.csv")

In [7]:
df.head()

,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,existing_loans,current_emi_amount,credit_score,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi
0,38,F,Married,Professional,82600,Private,0.9,Mid-size,Rented,20000.0,...,Yes,23700.0,660.0,303200.0,70200.0,Personal Loan EMI,850000.0,15,Not_Eligible,500.0
1,38,F,Married,Graduate,21500,Private,7.0,MNC,Family,0.0,...,Yes,4100.0,714.0,92500.0,26900.0,E-commerce Shopping EMI,128000.0,19,Not_Eligible,700.0
2,38,M,Married,Professional,86100,Private,5.8,Startup,Own,0.0,...,No,0.0,650.0,672100.0,324200.0,Education EMI,306000.0,16,Eligible,27775.0
3,58,F,Married,High School,66800,Private,2.2,Mid-size,Own,0.0,...,No,0.0,685.0,440900.0,178100.0,Vehicle EMI,304000.0,83,Eligible,16170.0
4,48,F,Married,Professional,57300,Private,3.4,Mid-size,Family,0.0,...,No,0.0,770.0,97300.0,28200.0,Home Appliances EMI,252000.0,7,Not_Eligible,500.0


In [8]:
df['total_expenses'] = (df['monthly_rent']+
                        df['other_monthly_expenses']+
                        df['school_fees']+
                        df['travel_expenses']+
                        df['groceries_utilities']+
                        df['college_fees'])
df['expenses_ratio'] = df['total_expenses'] / df['monthly_salary']
df['emi_burden'] = df['current_emi_amount'] / df['monthly_salary']

In [9]:
X = df.drop(columns=['emi_eligibility', 'max_monthly_emi'])
y_reg = df['max_monthly_emi']
y_class = df['emi_eligibility']

In [10]:
df.shape

(391003, 30)

In [11]:
df.head()

,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi,total_expenses,expenses_ratio,emi_burden
0,38,F,Married,Professional,82600,Private,0.9,Mid-size,Rented,20000.0,...,303200.0,70200.0,Personal Loan EMI,850000.0,15,Not_Eligible,500.0,59900.0,0.725182,0.286925
1,38,F,Married,Graduate,21500,Private,7.0,MNC,Family,0.0,...,92500.0,26900.0,E-commerce Shopping EMI,128000.0,19,Not_Eligible,700.0,15400.0,0.716279,0.190698
2,38,M,Married,Professional,86100,Private,5.8,Startup,Own,0.0,...,672100.0,324200.0,Education EMI,306000.0,16,Eligible,27775.0,35600.0,0.413473,0.000000
3,58,F,Married,High School,66800,Private,2.2,Mid-size,Own,0.0,...,440900.0,178100.0,Vehicle EMI,304000.0,83,Eligible,16170.0,37400.0,0.559880,0.000000
4,48,F,Married,Professional,57300,Private,3.4,Mid-size,Family,0.0,...,97300.0,28200.0,Home Appliances EMI,252000.0,7,Not_Eligible,500.0,58600.0,1.022688,0.000000


In [12]:
# test-train split
X_train_reg, x_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=11)

In [13]:
# encoding 
gender_map = {
    'F' : '0',
    'M' : '1'
}

marital_map = {
    'Married' : '0',
    'Single' : '1'
}

education_map = {
    'High School' : '0',
    'Graduate' : '1',
    'Post Graduate' : '2',
    'Professional' : '3'
}

employment_map = {
    'Self-employed' : '0',
    'Private' : '1',
    'Government' : '2'
}

company_map = {
    'Startup' : '0',
    'Small' : '1',
    'Mid-size' : '2',
    'Large Indian' : '3',
    'MNC' : '4'
}

house_map = {
    'Rented' : '0',
    'Family' : '1',
    'Own' : '2'
}

existing_loan_map = {
    'No' : '0',
    'Yes' : '1'
}

emi_scenario_map = {
    'Personal Loan EMI' : '0',
    'E-commerce Shopping EMI' : '1',
    'Education EMI' : '2',
    'Vehicle EMI' : '3',
    'Home Appliances EMI' : '4'
}
def encoder(df):
    df['gender'] = df['gender'].replace(gender_map).astype('int')
    df['marital_status'] = df['marital_status'].replace(marital_map).astype('int')
    df['education'] = df['education'].replace(education_map).astype('int')
    df['employment_type'] = df['employment_type'].replace(employment_map).astype('int')
    df['company_type'] = df['company_type'].replace(company_map).astype('int')
    df['house_type'] = df['house_type'].replace(house_map).astype('int')
    df['existing_loans'] = df['existing_loans'].replace(existing_loan_map).astype('int')
    df['emi_scenario'] = df['emi_scenario'].replace(emi_scenario_map).astype('int')

    return df

X_train_reg_final = encoder(df=X_train_reg)
x_test_reg_final = encoder(df=x_test_reg)

In [14]:
# Regression Models

In [15]:
lr = LinearRegression()
lr.fit(X_train_reg_final, y_train_reg)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [16]:
lr.coef_

array([ 2.46228066e+00, -6.38657854e+00, -7.38739764e+00,  4.19928449e+02,
        2.16019437e-02,  6.78138396e+02,  3.68553301e+01,  8.84851638e+00,
        5.08892171e+02, -2.59206434e-01, -2.22427506e+01, -2.22427506e+01,
       -2.81500135e-01, -2.30596561e-01,  3.49628386e-01,  3.58487884e-01,
        4.31248303e-02, -1.89675087e+03, -4.00241340e-01,  7.81983582e+00,
        1.05096342e-02,  4.18709095e-04,  1.81950819e+00, -3.81554143e-05,
        7.19391842e-01, -2.00620362e-02, -3.58924803e+02,  3.34186263e+03])

In [17]:
pred = lr.predict(x_test_reg_final)

In [18]:
r2_score(y_test_reg, pred)

0.7151133811579604

In [19]:
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    n_jobs=-1
)

In [20]:
rf.fit(X_train_reg_final, y_train_reg)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [21]:
pred = rf.predict(x_test_reg_final)

In [22]:
r2_score(y_test_reg, pred)

0.984090785796085

In [27]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test_reg, pred)

In [28]:
xgb = XGBRegressor(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    tree_method='hist'
)
xgb.fit(X_train_reg_final, y_train_reg)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [29]:
pred = xgb.predict(x_test_reg_final)

In [30]:
r2_score(y_test_reg, pred)

0.99123365608746

In [31]:
mean_squared_error(y_test_reg, pred)

534070.5213977628

In [32]:
root_mean_squared_error(y_test_reg, pred)

730.8012872168213

In [33]:
mean_absolute_percentage_error(y_test_reg, pred)

0.07797230669878685

In [34]:
score = cross_val_score(xgb, X_train_reg_final, y_train_reg, cv=5, scoring='r2')
print(f"CV R2 score: {score.mean()}")

CV R2 score: 0.990452396081236


In [ ]:
# Classification Models

In [35]:
X_train_class, x_test_class,  y_train_class, y_test_class = train_test_split(X, y_class, test_size=0.2, random_state=11)

In [36]:
# encoding 
gender_map = {
    'F' : '0',
    'M' : '1'
}

marital_map = {
    'Married' : '0',
    'Single' : '1'
}

education_map = {
    'High School' : '0',
    'Graduate' : '1',
    'Post Graduate' : '2',
    'Professional' : '3'
}

employment_map = {
    'Self-employed' : '0',
    'Private' : '1',
    'Government' : '2'
}

company_map = {
    'Startup' : '0',
    'Small' : '1',
    'Mid-size' : '2',
    'Large Indian' : '3',
    'MNC' : '4'
}

house_map = {
    'Rented' : '0',
    'Family' : '1',
    'Own' : '2'
}

existing_loan_map = {
    'No' : '0',
    'Yes' : '1'
}

emi_scenario_map = {
    'Personal Loan EMI' : '0',
    'E-commerce Shopping EMI' : '1',
    'Education EMI' : '2',
    'Vehicle EMI' : '3',
    'Home Appliances EMI' : '4'
}
def encoder(df):
    df['gender'] = df['gender'].replace(gender_map).astype('int')
    df['marital_status'] = df['marital_status'].replace(marital_map).astype('int')
    df['education'] = df['education'].replace(education_map).astype('int')
    df['employment_type'] = df['employment_type'].replace(employment_map).astype('int')
    df['company_type'] = df['company_type'].replace(company_map).astype('int')
    df['house_type'] = df['house_type'].replace(house_map).astype('int')
    df['existing_loans'] = df['existing_loans'].replace(existing_loan_map).astype('int')
    df['emi_scenario'] = df['emi_scenario'].replace(emi_scenario_map).astype('int')

    return df

X_train_class_encoded = encoder(df=X_train_class)
x_test_class_encoded = encoder(df=x_test_class)

In [37]:
X_train_class_encoded.shape

(312802, 28)

In [38]:
log = LogisticRegression()
log.fit(X_train_class_encoded, y_train_class)

c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [39]:
log.coef_

array([[-2.22084311e-07, -4.09864577e-09, -2.07271654e-09,
         5.91581630e-09,  2.66742415e-06, -6.69820198e-10,
         1.61008872e-08, -1.59000413e-08,  5.66908728e-09,
        -4.57723789e-05, -2.00846750e-08, -1.32764342e-08,
        -5.48762065e-05, -5.12631725e-05,  4.67939338e-05,
         7.55291358e-05,  2.44890731e-05, -1.42151644e-08,
        -8.47845001e-05, -2.46044358e-06,  1.66855480e-06,
        -1.25990977e-07, -4.60063869e-09, -5.10781773e-06,
         1.01533787e-06, -5.09961519e-06, -1.08864033e-08,
        -2.46260340e-09],
       [-5.66598562e-07, -8.98851602e-09, -4.75295473e-09,
        -9.29925140e-09,  3.41650600e-07, -1.56749653e-08,
        -6.85520882e-08, -3.63712346e-08, -1.44833142e-08,
        -1.63430891e-06, -4.14726011e-08, -2.64996248e-08,
        -2.90378624e-05, -1.54920575e-05,  4.27930268e-06,
         1.20081279e-05,  5.66975973e-06, -8.35055305e-09,
        -2.98689744e-05, -9.89772500e-06, -2.40730028e-07,
         1.31905031e-06, -3.07

In [40]:
pred = log.predict(x_test_class)

In [41]:
accuracy_score(y_test_class, pred)

0.8582754696231506

In [42]:
precision_score(y_test_class, pred, average='weighted')

c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


0.8194448583098307

In [43]:
f1_score(y_test_class, pred, average='weighted')

0.838305837103829

In [44]:
recall_score(y_test_class, pred, average='weighted')

0.8582754696231506

In [45]:
rf_class = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
)
rf_class.fit(X_train_class_encoded, y_train_class)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [46]:
pred = rf_class.predict(x_test_class_encoded)

In [47]:
accuracy_score(y_test_class, pred)

0.9096430991931049

In [48]:
precision_score(y_test_class, pred, average='weighted', zero_division=0)

0.8693837257084782

In [49]:
f1_score(y_test_class, pred, average='weighted')

0.8881808624720235

In [50]:
recall_score(y_test_class, pred, average='weighted')

0.9096430991931049

In [51]:
lc = LabelEncoder()
y_train_class_encoded = lc.fit_transform(y_train_class)
y_test_class_encoded = lc.transform(y_test_class)

In [52]:
xgb_class = XGBClassifier(
    n_estimators=50,
    max_depth=4,
    n_jobs=-1
)

In [53]:
xgb_class.fit(X_train_class_encoded, y_train_class_encoded)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [54]:
pred = xgb_class.predict(x_test_class_encoded)

In [55]:
pred_prob = xgb_class.predict_proba(x_test_class_encoded)

In [56]:
pred

array([0, 2, 2, ..., 2, 0, 0], shape=(78201,))

In [57]:
roc_auc_score(y_test_class_encoded, pred_prob,multi_class='ovo', average='weighted')

np.float64(0.9776399024160394)

In [58]:
accuracy_score(y_test_class_encoded, pred)

0.9463689722637818

In [59]:
f1_score(y_test_class_encoded, pred, average='weighted')

0.9256552467603655

In [60]:
recall_score(y_test_class_encoded, pred, average='weighted')

0.9463689722637818

In [61]:
precision_score(y_test_class_encoded, pred, average='weighted')

0.9088698539983783

In [62]:
models_reg = [
    (
        "Linear Regression Model",
        {},
        LinearRegression(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),

    (
        "Random Forest Regressor",
        {"n_estimators":50, "max_depth":15, "n_jobs":-1},
        RandomForestRegressor(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),

    (
        "XGBoost Regressor",
        {"n_estimators":50, "max_depth":15, "n_jobs":-1, "tree_method":'hist'},
        XGBRegressor(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),
]

models_class = [
    (
        "Logistic Regression",
        {},
        LogisticRegression(),
        (X_train_class_encoded, y_train_class),
        (x_test_class_encoded,y_test_class)
    ),

    (
        "Random Forest Classifier",
        {"n_estimators":50, "max_depth":10, "n_jobs":-1},
        RandomForestClassifier(),
        (X_train_class_encoded, y_train_class),
        (x_test_class_encoded,y_test_class)
    ),

    (
        "XGBoost Classifier",
        {"n_estimators":50, "max_depth":10, "n_jobs":-1},
        XGBClassifier(),
        (X_train_class_encoded, y_train_class_encoded),
        (x_test_class_encoded,y_test_class_encoded)
    )
]

In [63]:
report_reg = []
for model_name, params, model, train_set, test_set in models_reg:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)

    # calculate metrics
    rmse = root_mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    mape = mean_absolute_percentage_error(y_test, pred)

    # store the metrics
    report_reg.append((model_name, rmse, mae, r2, mape))

In [64]:
report_class = []
for model_name, params, model, train_set, test_set in models_class:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)
    pred_prob = model.predict_proba(x_test)

    # calculate metrics
    precision = precision_score(y_test, pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test, pred)
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    roc_auc = roc_auc_score(y_test_class_encoded, pred_prob,multi_class='ovo', average='weighted')
    roc_auc = float(roc_auc)
    # store the metrics
    report_class.append((model_name, accuracy, precision, recall, f1, roc_auc))

c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [65]:
report_reg

[('Linear Regression Model',
  4166.0653376301825,
  2982.7649824663263,
  0.7151133811579604,
  1.9078289862391105),
 ('Random Forest Regressor',
  980.9686709666313,
  418.1517334394038,
  0.984204616583703,
  0.0756087290815887),
 ('XGBoost Regressor',
  841.295980871594,
  311.1127064884015,
  0.9883823704111818,
  0.06412800824788314)]

In [66]:
report_class

[('Logistic Regression',
  0.8582754696231506,
  0.8194448583098307,
  0.8582754696231506,
  0.838305837103829,
  0.8388364946695954),
 ('Random Forest Classifier',
  0.9078144780757279,
  0.8675811481833083,
  0.9078144780757279,
  0.8862343414295715,
  0.9373635256407282),
 ('XGBoost Classifier',
  0.9617651948184806,
  0.956245440875611,
  0.9617651948184806,
  0.9568478745195503,
  0.9883543951900988)]

In [67]:
mlflow.set_experiment("First Classification Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(models_class):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = report_class[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({"Accuracy": report[1],
                            "Precision": report[2],
                            "Recall": report[3],
                            "F1": report[4],
                            "ROC_AUC": report[5]})
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/04/27 16:24:20 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/04/27 16:24:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instea

🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/1/runs/0e6a7dc1ebb04fa3ba43fa6e943da147
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/04/27 16:24:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 16:24:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/1/runs/445c7edd89d0488a8d2bb79d8e5ea396
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/04/27 16:24:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost Classifier at: http://127.0.0.1:5000/#/experiments/1/runs/11cc622207d84b8c986f9aef15329b91
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Our first experiments had unscaled data for linear regression and logistic regression, which is not the correct way to train these models.
And for logistic regression, we've to handle imbalance in data
Also, we have not tuned hyper-parameters.
Let's do what we have not done.
Re-read the data for updated models

In [68]:
# Data Loading and Required Preprocessing and Feature Engineering
df = pd.read_csv("cleaned_emi_prediction_dataset.csv")

# feature engineering- add some calculated featires
df['total_expense'] = (df['monthly_rent'] + 
                       df['college_fees'] + 
                       df['school_fees'] +
                       df['travel_expenses'] +
                       df['groceries_utilities'] + 
                       df['other_monthly_expenses'])
df['expense_ratio'] = df['total_expense'] / df['monthly_salary']
df['emi_burden'] = df['current_emi_amount'] / df['monthly_salary']

# setting feature and target
X = df.drop(columns=['emi_eligibility', 'max_monthly_emi'])
y_reg = df['max_monthly_emi']
y_class = df['emi_eligibility']

# Regression Split
X_train_reg, x_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=11)
# Encoding
# encoding 
gender_map = {
    'F' : '0',
    'M' : '1'
}

marital_map = {
    'Married' : '0',
    'Single' : '1'
}

education_map = {
    'High School' : '0',
    'Graduate' : '1',
    'Post Graduate' : '2',
    'Professional' : '3'
}

employment_map = {
    'Self-employed' : '0',
    'Private' : '1',
    'Government' : '2'
}

company_map = {
    'Startup' : '0',
    'Small' : '1',
    'Mid-size' : '2',
    'Large Indian' : '3',
    'MNC' : '4'
}

house_map = {
    'Rented' : '0',
    'Family' : '1',
    'Own' : '2'
}

existing_loan_map = {
    'No' : '0',
    'Yes' : '1'
}

emi_scenario_map = {
    'Personal Loan EMI' : '0',
    'E-commerce Shopping EMI' : '1',
    'Education EMI' : '2',
    'Vehicle EMI' : '3',
    'Home Appliances EMI' : '4'
}
def encoder(df):
    df['gender'] = df['gender'].replace(gender_map).astype('int')
    df['marital_status'] = df['marital_status'].replace(marital_map).astype('int')
    df['education'] = df['education'].replace(education_map).astype('int')
    df['employment_type'] = df['employment_type'].replace(employment_map).astype('int')
    df['company_type'] = df['company_type'].replace(company_map).astype('int')
    df['house_type'] = df['house_type'].replace(house_map).astype('int')
    df['existing_loans'] = df['existing_loans'].replace(existing_loan_map).astype('int')
    df['emi_scenario'] = df['emi_scenario'].replace(emi_scenario_map).astype('int')

    return df

X_train_reg_final = encoder(df=X_train_reg)
x_test_reg_final = encoder(df=x_test_reg)

# Scaling for Linear Regression model
scaler = StandardScaler()
X_train_reg_final_scaled = scaler.fit_transform(X_train_reg_final)
x_test_reg_final_scaled = scaler.transform(x_test_reg_final)

In [69]:
# Classification Split
X_train_class, x_test_class,  y_train_class, y_test_class = train_test_split(X, y_class, test_size=0.2, random_state=11)

# Encoding
# encoding 
gender_map = {
    'F' : '0',
    'M' : '1'
}

marital_map = {
    'Married' : '0',
    'Single' : '1'
}

education_map = {
    'High School' : '0',
    'Graduate' : '1',
    'Post Graduate' : '2',
    'Professional' : '3'
}

employment_map = {
    'Self-employed' : '0',
    'Private' : '1',
    'Government' : '2'
}

company_map = {
    'Startup' : '0',
    'Small' : '1',
    'Mid-size' : '2',
    'Large Indian' : '3',
    'MNC' : '4'
}

house_map = {
    'Rented' : '0',
    'Family' : '1',
    'Own' : '2'
}

existing_loan_map = {
    'No' : '0',
    'Yes' : '1'
}

emi_scenario_map = {
    'Personal Loan EMI' : '0',
    'E-commerce Shopping EMI' : '1',
    'Education EMI' : '2',
    'Vehicle EMI' : '3',
    'Home Appliances EMI' : '4'
}
def encoder(df):
    df['gender'] = df['gender'].replace(gender_map).astype('int')
    df['marital_status'] = df['marital_status'].replace(marital_map).astype('int')
    df['education'] = df['education'].replace(education_map).astype('int')
    df['employment_type'] = df['employment_type'].replace(employment_map).astype('int')
    df['company_type'] = df['company_type'].replace(company_map).astype('int')
    df['house_type'] = df['house_type'].replace(house_map).astype('int')
    df['existing_loans'] = df['existing_loans'].replace(existing_loan_map).astype('int')
    df['emi_scenario'] = df['emi_scenario'].replace(emi_scenario_map).astype('int')

    return df

X_train_class_encoded = encoder(df=X_train_class)
x_test_class_encoded = encoder(df=x_test_class)

# Target Encoding for calssification Model
lc = LabelEncoder()
y_train_class_encoded = lc.fit_transform(y_train_class)
y_test_class_encoded = lc.transform(y_test_class)

In [70]:
# Imabalance handling using SMOTE for classification models
smote = SMOTE(random_state=11)
X_train_class_encoded, y_train_class_encoded = smote.fit_resample(X_train_class_encoded, y_train_class_encoded)

In [71]:
# scaling data for Logistic regression

scaler = StandardScaler()
X_train_class_encoded_scaled = scaler.fit_transform(X_train_class_encoded)
x_test_class_encoded_scaled = scaler.transform(x_test_class_encoded)

In [ ]:
# Updated Regression Models

In [72]:
# Scale data for Linear Regression

scaler = StandardScaler()
X_train_reg_final_scaled = scaler.fit_transform(X_train_reg_final)
x_test_reg_final_scaled = scaler.transform(x_test_reg_final)

In [73]:
lr_scaled = LinearRegression()
lr_scaled.fit(X_train_reg_final_scaled, y_train_reg)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [74]:
pred = lr_scaled.predict(x_test_reg_final_scaled)

In [75]:
r2_score(y_test_reg, pred)

0.715113381157926

In [76]:
param_grid_xgb_reg = {
    "n_estimators" : [50,70,100],
    "max_depth" : [10,15],
    "tree_method" : ['hist'],
    "n_jobs" : [-1]
}

grid_search = GridSearchCV(XGBRegressor(), param_grid_xgb_reg, cv=5, scoring='r2')
grid_search.fit(X_train_reg_final, y_train_reg)

print(grid_search.best_score_)
grid_search.best_params_

0.9907198151828884


{'max_depth': 10, 'n_estimators': 100, 'n_jobs': -1, 'tree_method': 'hist'}

In [ ]:
# Updated Classification Models

In [77]:
# since we have imbalane present in our data, we need to handle it for classification models

smote = SMOTE(random_state=11)
X_train_class_encoded, y_train_class_encoded = smote.fit_resample(X_train_class_encoded, y_train_class_encoded)

In [78]:
X_train_class_encoded.shape

(725130, 28)

In [79]:
# scaling data for logistic regression

scaler = StandardScaler()
X_train_class_encoded_scaled = scaler.fit_transform(X_train_class_encoded)
x_test_class_encoded_scaled = scaler.transform(x_test_class_encoded)

In [80]:
log_scaled = LogisticRegression()
log_scaled.fit(X_train_class_encoded_scaled, y_train_class_encoded)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [81]:
p = log_scaled.predict(x_test_class_encoded_scaled)

In [82]:
p

array([0, 2, 2, ..., 2, 0, 0], shape=(78201,))

In [83]:
accuracy_score(y_test_class_encoded, p)

0.8196570376337898

In [84]:
rf_class = RandomForestClassifier(
    n_estimators=250,
    max_depth=20,
    n_jobs=-1,
)
rf_class.fit(X_train_class_encoded, y_train_class_encoded)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",250
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [85]:
p2 = rf_class.predict(x_test_class_encoded)

In [86]:
accuracy_score(y_test_class_encoded, p2)

0.9168041329394765

In [87]:
recall_score(y_test_class_encoded, p2, average='weighted')

0.9168041329394765

In [88]:
f1_score(y_test_class_encoded, p2, average='weighted')

0.9229236192569137

In [89]:
precision_score(y_test_class_encoded, p2, average='weighted')

0.930467542593588

In [90]:
xgb_class = XGBClassifier(
    n_estimators=150,
    max_depth = 15,
    tree_method = 'hist',
    n_jobs=-1
)

xgb_class.fit(X_train_class_encoded, y_train_class_encoded)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [91]:
pred = xgb_class.predict(x_test_class_encoded)

In [92]:
accuracy_score(y_test_class_encoded, pred)

0.9570593726422936

In [ ]:
# Implementing MLflow for models

In [93]:
# Final Models

# In regression models we are now using scaled data and using tuned hyper-parameters
final_models_reg = [
    (
        "Linear Regression Model",
        {},
        LinearRegression(),
        (X_train_reg_final_scaled, y_train_reg),
        (x_test_reg_final_scaled, y_test_reg)
    ),  # In Linear Regression Model, we have updated the model with scaled feature data

    (
        "Random Forest Regressor",
        {"n_estimators":60, "max_depth":20, "n_jobs":-1},
        RandomForestRegressor(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),  # In Random Forest Regressor Model, we have tuned for hyper-parameters

    (
        "XGBoost Regressor",
        {"n_estimators":100, "max_depth":10, "n_jobs":-1, "tree_method":'hist'},
        XGBRegressor(),
        (X_train_reg_final, y_train_reg),
        (x_test_reg_final, y_test_reg)
    ),  # In XGBoost Regressor Model, we have tuned for hyper-parameters
]

# In classification models we are using balanced and scaled data and using tuned hyper-parameters 
# (scaled data only for logit, rf and xgb dont need scaled data)
final_models_class = [
    (
        "Logistic Regression",
        {},
        LogisticRegression(),
        (X_train_class_encoded_scaled, y_train_class_encoded),
        (x_test_class_encoded_scaled, y_test_class_encoded)
    ),  # In Logistic Regression Model, we have updated the model with balanced and scaled data

    (
        "Random Forest Classifier",
        {"n_estimators":100, "max_depth":20, "n_jobs":-1},
        RandomForestClassifier(),
        (X_train_class_encoded, y_train_class_encoded),
        (x_test_class_encoded, y_test_class_encoded)
    ),  # In Random Forest Classifier Model, we have now used balanced data and best hyper-parameters

    (
        "XGBoost Classifier",
        {"n_estimators":150, "max_depth":15, "n_jobs":-1, "tree_method":'hist'},
        XGBClassifier(),
        (X_train_class_encoded, y_train_class_encoded),
        (x_test_class_encoded,y_test_class_encoded)
    )   # In XGBoost Classifier Model, we have now used balanced data and best hyper-parameters
]

In [ ]:
# Training loops and scoring metrics

In [94]:
final_report_reg = []
for model_name, params, model, train_set, test_set in final_models_reg:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)

    # calculate metrics
    rmse = root_mean_squared_error(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    mape = mean_absolute_percentage_error(y_test, pred)

    # store the metrics
    final_report_reg.append((model_name, rmse, mae, r2, mape))

In [95]:
final_report_reg

[('Linear Regression Model',
  4166.065337630434,
  2982.764982464639,
  0.715113381157926,
  1.907828986234899),
 ('Random Forest Regressor',
  931.2942856135946,
  360.9747697781213,
  0.9857638100686672,
  0.06803893743098609),
 ('XGBoost Regressor',
  719.3522666439823,
  289.9341154186708,
  0.9915061784995349,
  0.07682763973603829)]

In [96]:
# save regression metrics
columns = ["Model", "RMSE", "MAE", "R2", "MAPE"]
reg_met_df = pd.DataFrame(final_report_reg, columns=columns)
reg_met_df.to_csv("reg_metrics.csv", index=False)

In [97]:
final_report_class = []
for model_name, params, model, train_set, test_set in final_models_class:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)
    pred_prob = model.predict_proba(x_test)

    # calculate metrics
    precision = precision_score(y_test, pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test, pred)
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    roc_auc = roc_auc_score(y_test_class_encoded, pred_prob,multi_class='ovo', average='weighted')
    roc_auc = float(roc_auc)
    # store the metrics
    final_report_class.append((model_name, accuracy, precision, recall, f1, roc_auc))

In [98]:
# save classification metrics
columns = ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]
class_met_df = pd.DataFrame(final_report_class, columns=columns)
class_met_df.to_csv("class_metrics.csv", index=False)

In [99]:
final_report_class

[('Logistic Regression',
  0.8196570376337898,
  0.8869020128954429,
  0.8196570376337898,
  0.8484310579134627,
  0.8944051890780014),
 ('Random Forest Classifier',
  0.9135688801933479,
  0.9280632365103196,
  0.9135688801933479,
  0.9200731914373846,
  0.9592707998392689),
 ('XGBoost Classifier',
  0.9570593726422936,
  0.9552393396389292,
  0.9570593726422936,
  0.9560826960842744,
  0.9867936959998926)]

In [ ]:
# update to mlflow

In [100]:
mlflow.set_experiment("Second Regression Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(final_models_reg):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = final_report_reg[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({"RMSE": report[1],
                            "MAE": report[2],
                            "R2": report[3],
                            "MAPE": report[4]})
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/04/27 16:58:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 16:58:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Linear Regression Model at: http://127.0.0.1:5000/#/experiments/2/runs/d919ec33c1b04e11b1bc7fc422f92be1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/04/27 16:58:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 16:58:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest Regressor at: http://127.0.0.1:5000/#/experiments/2/runs/bf551d6f6be549528c727352c6024dd0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/04/27 17:06:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost Regressor at: http://127.0.0.1:5000/#/experiments/2/runs/3a6fa4c658014fc6a2fcae90b7a59d84
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [101]:
mlflow.set_experiment("Second Classification Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(final_models_class):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = final_report_class[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({"Accuracy": report[1],
                            "Precision": report[2],
                            "Recall": report[3],
                            "F1": report[4],
                            "ROC_AUC": report[5]})
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model", pip_requirements=[])
        elif model_name=="Random Forest Classifier":
            mlflow.sklearn.log_model(model, "model", pip_requirements=[])
        else:
            mlflow.sklearn.log_model(model, "model")

2026/04/27 17:07:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 17:07:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/3/runs/3a83ab673e52439eaafcfdfd398ec31f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/04/27 17:07:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 17:07:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/3/runs/6a1c9a02eecf4e3f81d64d5865efe8ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/04/27 17:10:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost Classifier at: http://127.0.0.1:5000/#/experiments/3/runs/b519d01cb8304fb0821a1936cb6395c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [102]:
# rf model alone
rf_model = [
    (
        "Random Forest Classifier",
        {"n_estimators":100, "max_depth":20, "n_jobs":-1},
        RandomForestClassifier(),
        (X_train_class_encoded, y_train_class_encoded),
        (x_test_class_encoded, y_test_class_encoded)
    ),  # In Random Forest Classifier Model, we have now used balanced data and best hyper-parameters
]

In [103]:
report_rf = []
for model_name, params, model, train_set, test_set in rf_model:
    X_train = train_set[0]
    y_train = train_set[1]
    x_test = test_set[0]
    y_test = test_set[1]
    # apply hyper-parameters
    model.set_params(**params)
    # train model
    model.fit(X_train, y_train)
    # make predicion
    pred = model.predict(x_test)
    pred_prob = model.predict_proba(x_test)

    # calculate metrics
    precision = precision_score(y_test, pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_test, pred)
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    roc_auc = roc_auc_score(y_test_class_encoded, pred_prob,multi_class='ovo', average='weighted')
    roc_auc = float(roc_auc)
    # store the metrics
    report_rf.append((model_name, accuracy, precision, recall, f1, roc_auc))

In [104]:
report_rf

[('Random Forest Classifier',
  0.9138374189588369,
  0.927829572002557,
  0.9138374189588369,
  0.9201281246102945,
  0.9586307740812543)]

In [105]:
# Log RF Classifier

mlflow.set_experiment("Second Classification Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(rf_model):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = report_rf[i]
    if model_name=="Random Forest Classifier":
        with mlflow.start_run(run_name=model_name):
            mlflow.log_params(params)
            mlflow.log_metrics({"Accuracy": report[1],
                                "Precision": report[2],
                                "Recall": report[3],
                                "F1": report[4],
                                "ROC_AUC": report[5]})
            
            # if "XGB" in model_name:
                # mlflow.xgboost.log_model(model, "model", pip_requirements=[])
            if model_name=="Random Forest Classifier":
                mlflow.sklearn.log_model(model, "model", pip_requirements=[])
            # else:
                # mlflow.sklearn.log_model(model, "model")

2026/04/27 17:14:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 17:14:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/3/runs/ae8ed5a9a1f8479a85fe25abe422ee11
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [106]:
# Log XGB Classifier

mlflow.set_experiment("Second Classification Experiment")
mlflow.set_tracking_uri('http://127.0.0.1:5000')

for i, element in enumerate(rf_model):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = final_report_class[i]
    if "XGB" in model_name:
        with mlflow.start_run(run_name=model_name):
            mlflow.log_params(params)
            mlflow.log_metrics({"Accuracy": report[1],
                                "Precision": report[2],
                                "Recall": report[3],
                                "F1": report[4],
                                "ROC_AUC": report[5]})
            
            if "XGB" in model_name:
                mlflow.xgboost.log_model(model, "model", pip_requirements=[])
            # if model_name=="Random Forest Classifier":
                # mlflow.sklearn.log_model(model, "model", pip_requirements=[])
            # else:
                # mlflow.sklearn.log_model(model, "model")

In [ ]:
# Model Registration

In [107]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Second Regression Experiment")

with mlflow.start_run() as run:
    mlflow.log_param("model", "RandomForest")
    mlflow.log_metric("rmse", 10)

    print("Run ID:", run.info.run_id)

Run ID: 71c708eb8c58498cac519a6b70ec6a3d
🏃 View run nebulous-ray-377 at: http://127.0.0.1:5000/#/experiments/2/runs/71c708eb8c58498cac519a6b70ec6a3d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [108]:
# Linear Regression

import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Second Regression Experiment")

model = LinearRegression()

with mlflow.start_run() as run:
    model.fit(X_train, y_train)

    # ✅ THIS IS MANDATORY
    mlflow.sklearn.log_model(model, "model")

    run_id = run.info.run_id
    print("Run ID:", run_id)

2026/04/27 17:18:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 17:18:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: 96021d76ada54218b3ae94f7cd2b861d
🏃 View run aged-pig-139 at: http://127.0.0.1:5000/#/experiments/2/runs/96021d76ada54218b3ae94f7cd2b861d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [109]:
# Linear Regression

model_name = 'Linear Regression Model'
run_id = '4a10d51922c7449ca0d8f6212bb02eaf'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Regression Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'Linear Regression Model' already exists. Creating a new version of this model...
2026/04/27 17:18:13 WARNING mlflow.tracking._model_registry.fluent: Run with id 4a10d51922c7449ca0d8f6212bb02eaf has no artifacts at artifact path 'model', registering model based on models:/m-fc15a0e879454b63beb40cdbcd3e844b instead
2026/04/27 17:18:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Linear Regression Model, version 2
Created version '2' of model 'Linear Regression Model'.


🏃 View run dapper-lamb-548 at: http://127.0.0.1:5000/#/experiments/2/runs/4a10d51922c7449ca0d8f6212bb02eaf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [110]:
# Random Forest Regression

model_name = 'RF Regressor'
run_id = '4a10d51922c7449ca0d8f6212bb02eaf'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Regression Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'RF Regressor' already exists. Creating a new version of this model...
2026/04/27 17:18:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 4a10d51922c7449ca0d8f6212bb02eaf has no artifacts at artifact path 'model', registering model based on models:/m-fc15a0e879454b63beb40cdbcd3e844b instead
2026/04/27 17:18:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RF Regressor, version 2
Created version '2' of model 'RF Regressor'.


🏃 View run dapper-lamb-548 at: http://127.0.0.1:5000/#/experiments/2/runs/4a10d51922c7449ca0d8f6212bb02eaf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [111]:
# XGBoost Regression

model_name = 'XGBoost Regressor'
run_id = '4a10d51922c7449ca0d8f6212bb02eaf'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Regression Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'XGBoost Regressor' already exists. Creating a new version of this model...
2026/04/27 17:18:27 WARNING mlflow.tracking._model_registry.fluent: Run with id 4a10d51922c7449ca0d8f6212bb02eaf has no artifacts at artifact path 'model', registering model based on models:/m-fc15a0e879454b63beb40cdbcd3e844b instead
2026/04/27 17:18:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost Regressor, version 2
Created version '2' of model 'XGBoost Regressor'.


🏃 View run dapper-lamb-548 at: http://127.0.0.1:5000/#/experiments/2/runs/4a10d51922c7449ca0d8f6212bb02eaf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [112]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("Second Classification Experiment")

with mlflow.start_run() as run:
    model.fit(X_train, y_train)

    mlflow.sklearn.log_model(model, "model")

    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/model",
        name="Logistic Regression"
    )

2026/04/27 17:20:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 17:20:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Logistic Regression' already exists. Creating a new version of this model...
2026/04/27 17:20:23 WARNING mlflow.tracking._model_registry.fluent: Run with id 91f0563714884a0db3603ee624a4f790 has no artifacts at artifact path 'model', registering model based on models:/m-dad9ae9c2b6d4205b3a8ec3625b2b5f0 instead
2026/04/27 17:20:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Logistic Regressio

🏃 View run spiffy-penguin-131 at: http://127.0.0.1:5000/#/experiments/3/runs/91f0563714884a0db3603ee624a4f790
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [113]:
# Logistic Regression

model_name = 'Logistic Regression'
run_id = '41442bed2b544c1da5da9e3a0fb5b54d'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Classification Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'Logistic Regression' already exists. Creating a new version of this model...
2026/04/27 17:20:27 WARNING mlflow.tracking._model_registry.fluent: Run with id 41442bed2b544c1da5da9e3a0fb5b54d has no artifacts at artifact path 'model', registering model based on models:/m-46ffc3432f7b438da02cb78ad8b7ee70 instead
2026/04/27 17:20:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Logistic Regression, version 4
Created version '4' of model 'Logistic Regression'.


🏃 View run gifted-snipe-477 at: http://127.0.0.1:5000/#/experiments/3/runs/41442bed2b544c1da5da9e3a0fb5b54d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [114]:
# RF Classifier
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Second Classification Experiment")

model = RandomForestClassifier()

with mlflow.start_run() as run:
    model.fit(X_train, y_train)

    # ✅ Log model properly
    mlflow.sklearn.log_model(model, "model")

    # ✅ Register model
    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/model",
        name="RF Classifier"
    )

    print("Run ID:", run.info.run_id)

2026/04/27 17:31:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/27 17:31:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RF Classifier' already exists. Creating a new version of this model...
2026/04/27 17:46:55 WARNING mlflow.tracking._model_registry.fluent: Run with id c54de78a351f4673abb30ad4c3b1e6c0 has no artifacts at artifact path 'model', registering model based on models:/m-e739daef464444fbb993cbe7dad48080 instead
2026/04/27 17:46:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RF Classifier, version 3

Run ID: c54de78a351f4673abb30ad4c3b1e6c0
🏃 View run wise-dog-712 at: http://127.0.0.1:5000/#/experiments/3/runs/c54de78a351f4673abb30ad4c3b1e6c0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [115]:
# RF Classifier

model_name = 'RF Classifier'
run_id = '478ad12533374306b2e00a8120af10e3'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Classification Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'RF Classifier' already exists. Creating a new version of this model...
2026/04/27 17:59:01 WARNING mlflow.tracking._model_registry.fluent: Run with id 478ad12533374306b2e00a8120af10e3 has no artifacts at artifact path 'model', registering model based on models:/m-91eb777fc1674d20ad393584da3494ec instead
2026/04/27 17:59:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RF Classifier, version 4
Created version '4' of model 'RF Classifier'.


🏃 View run entertaining-kit-493 at: http://127.0.0.1:5000/#/experiments/3/runs/478ad12533374306b2e00a8120af10e3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [116]:
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier

mlflow.set_experiment("Second Classification Experiment")

model = XGBClassifier()

with mlflow.start_run() as run:
    model.fit(X_train, y_train)

    # ✅ Log model properly
    mlflow.xgboost.log_model(model, "model")

    # ✅ Register model
    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/model",
        name="XGBoost Classifier"
    )

    print("Run ID:", run.info.run_id)

2026/04/27 17:59:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'XGBoost Classifier' already exists. Creating a new version of this model...
2026/04/27 18:00:11 WARNING mlflow.tracking._model_registry.fluent: Run with id 19805920d7be4f15ab693bc7171dead2 has no artifacts at artifact path 'model', registering model based on models:/m-0227eb3da6344aff9f617a532ff72096 instead
2026/04/27 18:00:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost Classifier, version 3
Created version '3' of model 'XGBoost Classifier'.


Run ID: 19805920d7be4f15ab693bc7171dead2
🏃 View run adventurous-hound-91 at: http://127.0.0.1:5000/#/experiments/3/runs/19805920d7be4f15ab693bc7171dead2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [117]:
# XGBoost Classifier

model_name = 'XGBoost Classifier'
run_id = '27b56e9fbf5f4b99a344a7cd48f03729'
model_uri = f'runs:/{run_id}/model'

mlflow.set_experiment("Second Classification Experiment")
with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'XGBoost Classifier' already exists. Creating a new version of this model...
2026/04/27 18:00:17 WARNING mlflow.tracking._model_registry.fluent: Run with id 27b56e9fbf5f4b99a344a7cd48f03729 has no artifacts at artifact path 'model', registering model based on models:/m-71ed4be3687b465fbda09426b9afd227 instead
2026/04/27 18:00:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost Classifier, version 4
Created version '4' of model 'XGBoost Classifier'.


🏃 View run marvelous-finch-325 at: http://127.0.0.1:5000/#/experiments/3/runs/27b56e9fbf5f4b99a344a7cd48f03729
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


# Classification

In [118]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Second Classification Experiment")

for model_name, params, model, train_set, test_set in final_models_class:

    X_train, y_train = train_set
    X_test, y_test = test_set

    with mlflow.start_run(run_name=model_name) as run:

        # Train
        model.set_params(**params)
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        try:
            roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovo', average='weighted')
        except:
            roc_auc = 0.0

        # Log
        mlflow.log_params(params)
        mlflow.log_metrics({
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "ROC_AUC": roc_auc
        })

        # Log model (FIXED)
        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(model, name="model")
        else:
            mlflow.sklearn.log_model(model, name="model")

        # Register (CORRECT WAY)
        mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/model",
            name=model_name
        )

        print(f"✅ Done: {model_name}")

2026/04/27 18:03:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Logistic Regression' already exists. Creating a new version of this model...
2026/04/27 18:03:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 71246e75c2854d1a9cb7e05ad77685a5 has no artifacts at artifact path 'model', registering model based on models:/m-de0517d427944f81a214c013af3cecc0 instead
2026/04/27 18:03:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Logistic Regression, version 5
Created version '5' of model 'Logistic Regression'.


✅ Done: Logistic Regression
🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/3/runs/71246e75c2854d1a9cb7e05ad77685a5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/04/27 18:07:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Random Forest Classifier'.
2026/04/27 18:11:00 WARNING mlflow.tracking._model_registry.fluent: Run with id 7245dd4b13a043588e9de877b0bb8d09 has no artifacts at artifact path 'model', registering model based on models:/m-7f51a28cb76e448e8b9abd7dc16823ce instead
2026/04/27 18:11:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Random Forest Classifier, version 1
Created version '1' of model 'Random Forest Classifier'.


✅ Done: Random Forest Classifier
🏃 View run Random Forest Classifier at: http://127.0.0.1:5000/#/experiments/3/runs/7245dd4b13a043588e9de877b0bb8d09
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Registered model 'XGBoost Classifier' already exists. Creating a new version of this model...
2026/04/27 18:13:53 WARNING mlflow.tracking._model_registry.fluent: Run with id 3df213dfd1b44a4e920ab344a650f430 has no artifacts at artifact path 'model', registering model based on models:/m-52db0698325a4fdcb166ff60eceb50b2 instead
2026/04/27 18:13:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost Classifier, version 5
Created version '5' of model 'XGBoost Classifier'.


✅ Done: XGBoost Classifier
🏃 View run XGBoost Classifier at: http://127.0.0.1:5000/#/experiments/3/runs/3df213dfd1b44a4e920ab344a650f430
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


# Regression

In [119]:
mlflow.set_experiment("Second Regression Experiment")

for model_name, params, model, train_set, test_set in final_models_reg:

    X_train, y_train = train_set
    X_test, y_test = test_set

    with mlflow.start_run(run_name=model_name) as run:

        model.set_params(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        rmse = root_mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)

        mlflow.log_params(params)
        mlflow.log_metrics({
            "RMSE": rmse,
            "MAE": mae,
            "R2": r2,
            "MAPE": mape
        })

        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(model, name="model")
        else:
            mlflow.sklearn.log_model(model, name="model")

        mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/model",
            name=model_name
        )

        print(f"✅ Done: {model_name}")

2026/04/27 18:17:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'Linear Regression Model' already exists. Creating a new version of this model...
2026/04/27 18:17:32 WARNING mlflow.tracking._model_registry.fluent: Run with id 746f1306eb344c39b226c116df9c911d has no artifacts at artifact path 'model', registering model based on models:/m-86f9d5f24c8345b5a3d6b595588e2037 instead
2026/04/27 18:17:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Linear Regression Model, version 3
Created version '3' of model 'Linear Regression Model'.


✅ Done: Linear Regression Model
🏃 View run Linear Regression Model at: http://127.0.0.1:5000/#/experiments/2/runs/746f1306eb344c39b226c116df9c911d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/04/27 18:20:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Random Forest Regressor'.
2026/04/27 18:29:17 WARNING mlflow.tracking._model_registry.fluent: Run with id 40c39ac61a7f445aa901e98fc20c2ab4 has no artifacts at artifact path 'model', registering model based on models:/m-1908335795f448fd8cab141c105a3a5b instead
2026/04/27 18:29:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Random Forest Regressor, version 1
Created version '1' of model 'Random Forest Regressor'.


✅ Done: Random Forest Regressor
🏃 View run Random Forest Regressor at: http://127.0.0.1:5000/#/experiments/2/runs/40c39ac61a7f445aa901e98fc20c2ab4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


Registered model 'XGBoost Regressor' already exists. Creating a new version of this model...
2026/04/27 18:29:49 WARNING mlflow.tracking._model_registry.fluent: Run with id cb6ca1551ba64b219f76c54fcd908975 has no artifacts at artifact path 'model', registering model based on models:/m-c71ac7f3003441ad9b18a0959f8cc335 instead
2026/04/27 18:29:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost Regressor, version 3
Created version '3' of model 'XGBoost Regressor'.


✅ Done: XGBoost Regressor
🏃 View run XGBoost Regressor at: http://127.0.0.1:5000/#/experiments/2/runs/cb6ca1551ba64b219f76c54fcd908975
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
